# Data Cleaning and Preprocessing
Complete step-by-step pipeline for the `flights_sample_3m.csv` dataset.

## Step 0: Setup and Load Data

In [97]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

dtype_dict = {
    'DOT_CODE': 'int32',
    'FL_NUMBER': 'int32',
    'CRS_DEP_TIME': 'int32',
    'CRS_ARR_TIME': 'int32',
    'CANCELLED': 'float32',
    'DIVERTED': 'float32',
    'DISTANCE': 'float32'
}

print("Loading dataset...")
df = pd.read_csv('flights_sample_3m.csv', dtype=dtype_dict)
print(f"Initial Shape: {df.shape}")
df.head()


Loading dataset...
Initial Shape: (3000000, 32)


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,NaN,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,NaN,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


## Step 1: Remove Rows with Nulls in Critical Identity Columns
Only drop rows missing FL_DATE, AIRLINE, FL_NUMBER, ORIGIN, or DEST — fields needed to identify/route a flight. DEP_DELAY is intentionally null for cancelled flights (DQ-01) and must NOT be used as a drop criterion.

In [98]:
# Step 1: Remove rows where critical IDENTITY columns are missing
# PER DQ-01 / DQ-02: DEP_DELAY is intentionally null for cancelled flights.
# Dropping on DEP_DELAY would silently erase the entire cancelled-flight population.
# We only drop rows where the flight cannot be identified or routed at all.

critical_columns = ['FL_DATE', 'AIRLINE', 'FL_NUMBER', 'ORIGIN', 'DEST']

rows_before = len(df)
df = df.dropna(subset=critical_columns)
rows_dropped = rows_before - len(df)
print(f"Rows removed (unidentifiable records): {rows_dropped}")
print(f"Shape after Step 1: {df.shape}")

# Verify cancelled flights are still present
cancelled_count = int(df['CANCELLED'].sum())
print(f"Cancelled flights retained: {cancelled_count:,}  (should be ~79,140)")


Rows removed (unidentifiable records): 0
Shape after Step 1: (3000000, 32)
Cancelled flights retained: 79,140  (should be ~79,140)


## Step 2: Convert and Standardize Time Formats

In [99]:
# Step 2: Convert and standardize time formats
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# Extract departure hour (e.g., 1430 -> 14)
# PER DQ-01: cancelled flights have no departure time. Keep DEP_HOUR as NaN.
# DEP_TIME uses -1 as sentinel (not 0) because 0000 is a valid midnight departure time.
df['DEP_HOUR'] = (df['DEP_TIME'] // 100)        # stays float/nullable
df['DEP_TIME'] = df['DEP_TIME'].fillna(-1).astype('int32')  # -1 = not departed (sentinel)

print("Time formats standardized.")
df[['FL_DATE', 'DEP_TIME', 'DEP_HOUR']].head()


Time formats standardized.


,FL_DATE,DEP_TIME,DEP_HOUR
0,2019-01-09,1151,11.0
1,2022-11-19,2114,21.0
2,2022-07-22,1000,10.0
3,2023-03-06,1608,16.0
4,2020-02-23,1838,18.0


## Step 3: Remove Unwanted Characters from Text Columns


In [100]:
# Step 3: Remove unwanted characters from text columns
text_columns = ['AIRLINE', 'ORIGIN_CITY', 'DEST_CITY']
for col in text_columns:
    df[col] = df[col].astype(str).str.replace('"', '').str.strip()

print("Unwanted characters removed from text columns.")


Unwanted characters removed from text columns.


## Step 4: Handle Type Mismatches

In [101]:
# Step 4: Handle type mismatches (TRY_CAST equivalent)
numeric_columns = ['DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON',
                   'TAXI_IN', 'ARR_TIME', 'ARR_DELAY', 'ELAPSED_TIME', 'AIR_TIME']
for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Type mismatches handled successfully.")


Type mismatches handled successfully.


## Step 5: Remove Duplicate Records (Composite Key)

In [102]:
# Step 5: Remove duplicate records
# PER DQ report "Checks Performed": both exact and key-based duplicates
# were verified against the full dataset.

# Natural key duplicate check
composite_key = ['FL_DATE', 'AIRLINE', 'FL_NUMBER', 'ORIGIN', 'DEST']

rows_before = len(df)
df = df.drop_duplicates(subset=composite_key, keep='first')
key_dups_removed = rows_before - len(df)
print(f"Key-based duplicates removed: {key_dups_removed}")
print(f"Shape after Step 5: {df.shape}")

# Exact full-row duplicate check (all columns)
exact_dups = df.duplicated().sum()
print(f"Exact full-row duplicates remaining: {exact_dups}  (expected: 0)")


Key-based duplicates removed: 0
Shape after Step 5: (3000000, 33)
Exact full-row duplicates remaining: 0  (expected: 0)


## Step 6: Retain Structural Nulls
Strategy: Leave missing values in delay and timing columns untouched, as they represent legitimate operational states (cancellations/diversions).

In [103]:
# Step 6a: Inspect remaining missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_summary = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
print(missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing %', ascending=False))


                         Missing Count  Missing %
CANCELLATION_CODE              2920860      97.36
DELAY_DUE_LATE_AIRCRAFT        2466137      82.20
DELAY_DUE_CARRIER              2466137      82.20
DELAY_DUE_WEATHER              2466137      82.20
DELAY_DUE_NAS                  2466137      82.20
DELAY_DUE_SECURITY             2466137      82.20
ARR_DELAY                        86198       2.87
ELAPSED_TIME                     86198       2.87
AIR_TIME                         86198       2.87
WHEELS_ON                        79944       2.66
TAXI_IN                          79944       2.66
ARR_TIME                         79942       2.66
WHEELS_OFF                       78806       2.63
TAXI_OUT                         78806       2.63
DEP_DELAY                        77644       2.59
DEP_HOUR                         77615       2.59
CRS_ELAPSED_TIME                    14       0.00


In [104]:
# Step 6b: Retain structural nulls
# PER DQ-01, DQ-02, DQ-03: Do NOT impute DELAY_DUE_*, ARR_DELAY, or timing columns.
# These nulls encode actual cancellations, diversions, or flights that were not delayed enough to attribute.

remaining = df.isnull().sum().sum()
print(f"Total remaining missing values (expected due to structural nulls): {remaining}")
print(f"Shape: {df.shape}")


Total remaining missing values (expected due to structural nulls): 16062854
Shape: (3000000, 33)


## Step 7: Standardize Inconsistent Categories

In [105]:
# Step 7: Standardize inconsistent categories and handle time-format edge cases

# Title-case airline names
df['AIRLINE'] = df['AIRLINE'].str.strip().str.title()

# Uppercase airport codes
df['ORIGIN'] = df['ORIGIN'].str.strip().str.upper()
df['DEST'] = df['DEST'].str.strip().str.upper()

# Title-case city names
df['ORIGIN_CITY'] = df['ORIGIN_CITY'].str.strip().str.title()
df['DEST_CITY'] = df['DEST_CITY'].str.strip().str.title()

# Map cancellation codes to readable labels (leaving non-cancelled as NaN)
cancel_map = {
    'A': 'Carrier', 'B': 'Weather', 'C': 'NAS',
    'D': 'Security'
}
df['CANCELLATION_CODE'] = df['CANCELLATION_CODE'].map(cancel_map)

# PER DQ-07: Recode CRS_ARR_TIME = 2400 to 0000
mask_2400 = df['CRS_ARR_TIME'] == 2400
df.loc[mask_2400, 'CRS_ARR_TIME'] = 0
print(f"Recoded {mask_2400.sum()} rows with CRS_ARR_TIME=2400 to 0000")

print("Unique Airlines:", df['AIRLINE'].nunique())
print("Unique Origins:", df['ORIGIN'].nunique())
print("Cancellation Codes:", df['CANCELLATION_CODE'].unique())


Recoded 14 rows with CRS_ARR_TIME=2400 to 0000
Unique Airlines: 18
Unique Origins: 380
Cancellation Codes: [nan 'Security' 'Weather' 'Carrier' 'NAS']


## Step 8: Outlier Treatment (Flagging)
Strategy: Flag extreme values rather than capping them, as extreme delays are real events.

In [106]:
# Step 8: Outlier Treatment (Identify, do not cap)
# PER DQ-08: Do not remove or cap severe delays. Add a boolean severe_delay flag instead.

# Define severe delay as > 3 hours (180 mins)
# Use np.where to preserve NaN for cancelled/diverted flights (ARR_DELAY is null for them).
# Without this, .astype(float) silently converts NaN -> False -> 0.0, mislabelling cancelled
# flights as "not severely delayed" instead of "not applicable".
df['SEVERE_DELAY'] = np.where(df['ARR_DELAY'].isna(), np.nan, (df['ARR_DELAY'] > 180).astype(float))

severe_count = df['SEVERE_DELAY'].sum()
severe_null  = df['SEVERE_DELAY'].isna().sum()
print(f"Flagged {severe_count:.0f} flights as SEVERE_DELAY.")
print(f"SEVERE_DELAY = NaN (cancelled/diverted, not applicable): {severe_null:,}")
print(f"Shape after outlier treatment: {df.shape}")


Flagged 32263 flights as SEVERE_DELAY.
SEVERE_DELAY = NaN (cancelled/diverted, not applicable): 86,198
Shape after outlier treatment: (3000000, 34)


## Step 9: Datetime Feature Engineering
Extract Year, Month, Day, DayOfWeek, Quarter and time-of-day bins from FL_DATE.

In [107]:
# Step 9: Full datetime feature engineering

df['YEAR']        = df['FL_DATE'].dt.year
df['MONTH']       = df['FL_DATE'].dt.month
df['DAY']         = df['FL_DATE'].dt.day
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek   # 0=Monday, 6=Sunday
df['IS_WEEKEND']  = df['DAY_OF_WEEK'].isin([5, 6]).astype(int)
df['QUARTER']     = df['FL_DATE'].dt.quarter

# PER DQ-05: Add a derived partial_year flag for 2023
df['IS_PARTIAL_YEAR'] = (df['YEAR'] == 2023).astype(int)


# NOTE: DEP_TIME_OF_DAY is created in Step 10 with a NaN-safe get_time_of_day()
#       that returns 'Not Departed' for cancelled flights. Defined there only.

print("Datetime features created (including IS_PARTIAL_YEAR).")


Datetime features created (including IS_PARTIAL_YEAR).


## Step 10: Derived Features and KPIs
Create business-meaningful columns: IS_DELAYED, DELAY_SEVERITY, ROUTE, EFFICIENCY_RATIO, IS_PEAK_SEASON.

In [108]:
# Step 10: Create derived features and KPIs
import numpy as np

# KPI 1: IS_DELAYED - FAA standard (>15 min arrival delay)
# Use np.where to preserve NaN for cancelled/diverted flights.
# .astype(float) alone would convert NaN -> False -> 0.0, incorrectly marking
# cancelled flights as "on time". Pattern mirrors DELAY_SEVERITY_CAT below.
df['IS_DELAYED'] = np.where(df['ARR_DELAY'].isna(), np.nan, (df['ARR_DELAY'] > 15).astype(float))

# KPI 2: DELAY_SEVERITY_CAT - Categorical classification of delay
# Already NaN-safe via pd.isna() check inside classify_delay.
def classify_delay(delay):
    import pandas as pd
    if pd.isna(delay): return np.nan
    if delay <= 0:    return 'On Time / Early'
    elif delay <= 15: return 'Minor Delay'
    elif delay <= 60: return 'Moderate Delay'
    else:             return 'Severe Delay'

df['DELAY_SEVERITY_CAT'] = df['ARR_DELAY'].apply(classify_delay)

# KPI 3: ROUTE - Combined origin-destination for route-level analysis
df['ROUTE'] = df['ORIGIN'] + '-' + df['DEST']

# KPI 4: TOTAL_GROUND_TIME - Taxi out + Taxi in (non-flying time)
df['TOTAL_GROUND_TIME'] = df['TAXI_OUT'] + df['TAXI_IN']

# KPI 5: EFFICIENCY_RATIO - Actual vs Scheduled elapsed time (>1 = slower than planned)
df['EFFICIENCY_RATIO'] = (df['ELAPSED_TIME'] / df['CRS_ELAPSED_TIME']).round(3)

# KPI 6: IS_PEAK_SEASON - Summer (Jun-Aug) and Holiday (Nov-Dec)
df['IS_PEAK_SEASON'] = df['MONTH'].isin([6, 7, 8, 11, 12]).astype(int)

# KPI 7: DEP_TIME_OF_DAY - Bin departure hour into time-of-day buckets
# Must handle NaN DEP_HOUR (cancelled flights with no departure).
def get_time_of_day(hour):
    import pandas as pd
    if pd.isna(hour): return 'Not Departed'
    hour = int(hour)
    if hour < 6:    return 'Red-Eye'
    elif hour < 12: return 'Morning'
    elif hour < 17: return 'Afternoon'
    elif hour < 21: return 'Evening'
    else:           return 'Night'

df['DEP_TIME_OF_DAY'] = df['DEP_HOUR'].apply(get_time_of_day)

print("Derived Features and KPIs created.")
print(f"IS_DELAYED NaN count (cancelled/diverted): {df['IS_DELAYED'].isna().sum():,}")
print(f"DEP_TIME_OF_DAY 'Not Departed' count: {(df['DEP_TIME_OF_DAY'] == 'Not Departed').sum():,}")


Derived Features and KPIs created.
IS_DELAYED NaN count (cancelled/diverted): 86,198
DEP_TIME_OF_DAY 'Not Departed' count: 77,615


## Step 11: Categorical Encoding
Label-encode high-cardinality text columns for ML readiness. Original text columns are preserved for EDA.

In [109]:
# Step 11: Categorical encoding using Label Encoding (for ML readiness)
# We keep the original text columns for EDA and add encoded versions.

# PER DQ-01/DQ-02: Now that cancelled/diverted rows are retained,
# CANCELLATION_CODE, DEP_TIME_OF_DAY, and DELAY_SEVERITY_CAT will have
# genuine NaN values. .astype(str) would silently turn NaN -> "nan" string,
# creating a phantom category. We replace NaN with an explicit label first.
nan_fill_map = {
    'CANCELLATION_CODE': 'Not Cancelled',
    'DEP_TIME_OF_DAY':   'Not Departed',
    'DELAY_SEVERITY_CAT':'Not Applicable',
}

label_encode_cols = ['AIRLINE', 'ORIGIN', 'DEST', 'DEP_TIME_OF_DAY',
                     'DELAY_SEVERITY_CAT', 'CANCELLATION_CODE']

le = LabelEncoder()
for col in label_encode_cols:
    # Fill NaN with explicit label before encoding
    fill_val = nan_fill_map.get(col, 'Unknown')
    col_filled = df[col].fillna(fill_val).astype(str)
    encoded_col = col + '_ENC'
    df[encoded_col] = le.fit_transform(col_filled)
    print(f"{col} -> {encoded_col} | Categories: {list(le.classes_)[:5]}...")

print("Encoding complete.")


AIRLINE -> AIRLINE_ENC | Categories: ['Alaska Airlines Inc.', 'Allegiant Air', 'American Airlines Inc.', 'Delta Air Lines Inc.', 'Endeavor Air Inc.']...
ORIGIN -> ORIGIN_ENC | Categories: ['ABE', 'ABI', 'ABQ', 'ABR', 'ABY']...
DEST -> DEST_ENC | Categories: ['ABE', 'ABI', 'ABQ', 'ABR', 'ABY']...
DEP_TIME_OF_DAY -> DEP_TIME_OF_DAY_ENC | Categories: ['Afternoon', 'Evening', 'Morning', 'Night', 'Not Departed']...
DELAY_SEVERITY_CAT -> DELAY_SEVERITY_CAT_ENC | Categories: ['Minor Delay', 'Moderate Delay', 'Not Applicable', 'On Time / Early', 'Severe Delay']...
CANCELLATION_CODE -> CANCELLATION_CODE_ENC | Categories: ['Carrier', 'NAS', 'Not Cancelled', 'Security', 'Weather']...
Encoding complete.


## Step 12: Numerical Scaling (StandardScaler)
Normalize continuous features to mean=0, std=1. Required for distance-based models (KNN, SVM, Logistic Regression).

In [110]:
# # Step 12: Standard scaling for continuous numerical features
# # Mean=0, Std=1 - required for KNN, SVM, Logistic Regression etc.

# scale_cols = ['DEP_DELAY', 'ARR_DELAY', 'DISTANCE', 'AIR_TIME',
#               'TAXI_OUT', 'TAXI_IN', 'TOTAL_GROUND_TIME', 'EFFICIENCY_RATIO']

# scaler = StandardScaler()
# scaled_values = scaler.fit_transform(df[scale_cols])

# scaled_df = pd.DataFrame(
#     scaled_values,
#     columns=[c + '_SCALED' for c in scale_cols],
#     index=df.index
# )
# df = pd.concat([df, scaled_df], axis=1)

# print("Scaled columns added:")
# print(df[[c + '_SCALED' for c in scale_cols]].describe().round(3))


## Step 13: Drop Redundant / Low-Value Columns

In [111]:
# Step 13: Drop redundant / low-value columns
# PER DQ-04: Drop AIRLINE_DOT and DOT_CODE
cols_to_drop = ['AIRLINE_DOT', 'DOT_CODE']

# NOTE: We DO NOT drop high-missing columns here like the old code did,
# because DELAY_DUE_* columns are legitimately missing >80% of the time per DQ-03!
df = df.drop(columns=cols_to_drop, errors='ignore')

print(f"Dropped: {cols_to_drop}")
print(f"Final Dataset Shape: {df.shape}")


Dropped: ['AIRLINE_DOT', 'DOT_CODE']
Final Dataset Shape: (3000000, 52)


## Step 14: Save the Cleaned Dataset

In [112]:
# Step 14: Save the final cleaned dataset & delayed subset

# CSV for broad compatibility
df.to_csv('flights_cleaned.csv', index=False)
print("Saved: flights_cleaned.csv")

# Parquet for fast reloading (smaller size, preserves dtypes)
df.to_parquet('flights_cleaned.parquet', index=False)
print("Saved: flights_cleaned.parquet")

# PER DQ-03: Build delayed-flight subset using the EXACT definition from the report:
# "Filter to rows where DELAY_DUE_CARRIER is not null (i.e., the delayed subset)"
# This is the ~534K rows where FAA attributes delay causes (flights >15 min late).
# NOTE: ARR_DELAY > 0 (~978K) and ARR_DELAY not null (~2.9M) are DIFFERENT subsets.
df_delayed = df[df['DELAY_DUE_CARRIER'].notnull()].copy()
df_delayed.to_parquet('flights_delayed_subset.parquet', index=False)
print(f"Saved: flights_delayed_subset.parquet ({len(df_delayed):,} rows)")

print(f"Final clean dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")


Saved: flights_cleaned.csv
Saved: flights_cleaned.parquet
Saved: flights_delayed_subset.parquet (533,863 rows)
Final clean dataset: 3,000,000 rows x 52 columns
